In [ ]:
import argparse
import gzip
import io
import re
import csv
import sys
from typing import Optional


_RE_SMILES_Q = re.compile(r'SMILES\s*=\s*"([^"]*)"')
_RE_SMILES_NOQ = re.compile(r'SMILES\s*=\s*([^\s";]+)')
_RE_SID_Q = re.compile(r'SPECTRUMID\s*=\s*"([^"]*)"')
_RE_SID_NOQ = re.compile(r'SPECTRUMID\s*=\s*([^\s";]+)')

def extract_from_block(block_text: str) -> (Optional[str], Optional[str]):
    m = _RE_SMILES_Q.search(block_text)
    if m:
        smiles = m.group(1)
    else:
        m = _RE_SMILES_NOQ.search(block_text)
        smiles = m.group(1) if m else None

    m = _RE_SID_Q.search(block_text)
    if m:
        sid = m.group(1)
    else:
        m = _RE_SID_NOQ.search(block_text)
        sid = m.group(1) if m else None

    return smiles, sid

def open_maybe_gz(path: str):
    if path.endswith('.gz'):
        return io.TextIOWrapper(gzip.open(path, 'rb'), encoding='utf-8', errors='replace')
    else:
        return open(path, 'r', encoding='utf-8', errors='replace')

def process_mgf(inpath: str, outpath: str, progress_interval: int = 100000):
    in_f = open_maybe_gz(inpath)
    try:
        with in_f as fin, open(outpath, 'w', newline='', encoding='utf-8') as fout:
            writer = csv.writer(fout)
            writer.writerow(['SMILES', 'SPECTRUMID'])

            inside = False
            block_lines = []
            count = 0
            written = 0

            for lineno, raw_line in enumerate(fin, start=1):
                line = raw_line.rstrip('\n\r')
                if not inside:
                    if line.strip().upper() == 'BEGIN IONS':
                        inside = True
                        block_lines = [line]
                else:
                    block_lines.append(line)
                    if line.strip().upper() == 'END IONS':
                        block_text = "\n".join(block_lines)
                        count += 1
                        smiles, sid = extract_from_block(block_text)

                        writer.writerow([smiles or '', sid or ''])
                        written += 1

                        inside = False
                        block_lines = []

                        if progress_interval and (written % progress_interval == 0):
                            print(f'[进度] 已写入 {written:,} 条记录（处理到文件行 {lineno}）', file=sys.stderr)

            if inside and block_lines:
                block_text = "\n".join(block_lines)
                smiles, sid = extract_from_block(block_text)
                writer.writerow([smiles or '', sid or ''])
                written += 1

    except Exception as e:
        raise

def main():
    parser = argparse.ArgumentParser(description='从 MGF 中提取 SMILES 和 SPECTRUMID 到 CSV')
    parser.add_argument('-i', '--input', required=True, help='输入文件 (.mgf 或 .mgf.gz)')
    parser.add_argument('-o', '--output', required=True, help='输出 CSV 文件路径')
    parser.add_argument('--progress', type=int, default=100000,
                        help='每处理多少条打印一次进度（0 表示不打印），默认 100000')
    args = parser.parse_args()

    process_mgf(args.input, args.output, progress_interval=args.progress)

from pathlib import Path

input_path = r"ALL_GNPS.mgf" 
output_path = r"output.csv" 

process_mgf(input_path, output_path, progress_interval=100000)



[进度] 已写入 100,000 条记录（处理到文件行 33256999）
[进度] 已写入 200,000 条记录（处理到文件行 43583890）
[进度] 已写入 300,000 条记录（处理到文件行 50901592）
[进度] 已写入 400,000 条记录（处理到文件行 56602816）
[进度] 已写入 500,000 条记录（处理到文件行 61473074）
[进度] 已写入 600,000 条记录（处理到文件行 72697081）
[进度] 已写入 700,000 条记录（处理到文件行 90886119）
[进度] 已写入 800,000 条记录（处理到文件行 98172272）
[进度] 已写入 900,000 条记录（处理到文件行 105100806）
[进度] 已写入 1,000,000 条记录（处理到文件行 112401130）
[进度] 已写入 1,100,000 条记录（处理到文件行 119854749）
[进度] 已写入 1,200,000 条记录（处理到文件行 126545501）
[进度] 已写入 1,300,000 条记录（处理到文件行 134549599）
[进度] 已写入 1,400,000 条记录（处理到文件行 142177015）
[进度] 已写入 1,500,000 条记录（处理到文件行 148563055）
[进度] 已写入 1,600,000 条记录（处理到文件行 151847319）
[进度] 已写入 1,700,000 条记录（处理到文件行 156228541）
[进度] 已写入 1,800,000 条记录（处理到文件行 167046910）
完成：共解析到 1,850,113 个块，写入 1,850,113 条记录 -> "E:\LD\csv\output.csv"
